# Integrating Updated Price + Resale Data with CHAPA Applicant Data

This notebook performs the **final integration step** in the updated workflow:

1. Merge the **updated price + resale dataset** (`MergedPrice_Resale_Full.csv`)  
   with the **cleaned CHAPA applicant dataset** (`CHAPA_Chapter-40B_Application-Data_2021-2025_merged_v0.3.csv`)  
   to associate property and resale information with applicant details.

The output is a **fully merged dataset** ready for downstream analysis.

---

### Reference
- Updated Price + Resale Dataset: `MergedPrice_Resale_Full.csv`  
- Applicant Dataset: `CHAPA_Chapter-40B_Application-Data_2021-2025_merged_v0.3.csv`


In [ ]:
# ============================================================
# CHAPA Data Integration Pipeline
# Step: Merge Updated Price & Resale Data with Applicants
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Load Datasets
# ------------------------------------------------------------
def load_datasets():
    """
    Load the merged updated price + resale dataset
    and the cleaned applicant dataset.
    """
    merged_price_resale = pd.read_csv("/content/MergedPrice_Resale_Full.csv")
    applicants_df = pd.read_csv("/content/CHAPA_Chapter-40B_Application-Data_2021-2025_merged_v0.3.csv")
    return merged_price_resale, applicants_df

# ------------------------------------------------------------
# 2. Standardize Columns for Merging
# ------------------------------------------------------------
def standardize_columns(properties_df, applicants_df):
    """
    Standardize Town, Address, and Unit columns to lowercase stripped strings
    to ensure proper merge.
    """
    for df, town_col, addr_col, unit_col in [
        (properties_df, 'Town', 'Address', 'Unit Number'),
        (applicants_df, 'property_town_city', 'property_street_address', 'property_unit')
    ]:
        df['Town_clean'] = df[town_col].str.strip().str.lower()
        df['Address_clean'] = df[addr_col].str.strip().str.lower()
        df['Unit_clean'] = df[unit_col].astype(str).str.strip().str.lower()
    return properties_df, applicants_df

# ------------------------------------------------------------
# 3. Merge Properties with Applicants
# ------------------------------------------------------------
def merge_properties_applicants(properties_df, applicants_df):
    """
    Left join applicant data to properties dataset.
    Preserves all properties, even if no applicant exists.
    """
    merged = pd.merge(
        properties_df,
        applicants_df,
        on=['Town_clean', 'Address_clean', 'Unit_clean'],
        how='left',
        suffixes=('', '_applicant')
    )
    # Drop helper columns
    merged = merged.drop(columns=['Town_clean', 'Address_clean', 'Unit_clean'])
    return merged

# ------------------------------------------------------------
# 4. Save Final Dataset
# ------------------------------------------------------------
def save_dataset(df, filename="Final_Merged_CHAPA_Dataset.csv"):
    """Save fully merged dataset to CSV."""
    df.to_csv(filename, index=False)
    print(f"✅ Dataset saved as {filename}")

# ------------------------------------------------------------
# 5. Pipeline Execution
# ------------------------------------------------------------
def main():
    print("🚀 Starting final merge: Updated Price + Resale with Applicants...")
    merged_price_resale, applicants_df = load_datasets()
    merged_price_resale, applicants_df = standardize_columns(merged_price_resale, applicants_df)
    final_df = merge_properties_applicants(merged_price_resale, applicants_df)
    save_dataset(final_df)
    print("🏁 Final integration complete. Dataset ready for analysis.")

if __name__ == "__main__":
    main()
